该输出解析器允许用户以流行的 XML 格式从 LLM 获取结果。

请记住，大型语言模型是有漏洞的抽象！您必须使用具有足够容量的 LLM 来生成格式正确的 XML。




In [1]:
from langchain.output_parsers import XMLOutputParser
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
model = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)


In [3]:
actor_query = "生成汤姆·汉克斯的电影目录。"
output = model.invoke(
    f"""{actor_query}

响应以xml的结构返回，使用如下xml结构
```
<xml>
<movie>电影1</movie>
<movie>电影2</movie>
<xml>
``` 
"""
)
print(output.content)

<think>
好的，用户让我生成汤姆·汉克斯的电影目录，并且需要用XML格式返回。首先，我需要确定汤姆·汉克斯出演过哪些电影。他作为演员的作品很多，比如《当幸福来敲门》、《阿甘正传》、《百万英镑》、《无问西楼》等等。

接下来，我需要确保这些电影都是他主演的，并且是较为知名的。可能还需要考虑电影的年代和类型，这样在生成XML的时候能正确分类。然后，按照用户提供的结构，每个电影作为一个movie元素，包含title属性。

需要注意的是，用户要求用XML格式，所以必须正确使用标签。比如，每个movie标签内部有一个title属性，用双引号包裹。同时，确保XML的结构正确，没有语法错误。

另外，用户可能希望看到电影的名称和对应的标题，所以需要确认这些电影的正确名称，避免拼写错误。例如，《百万英镑》正确的英文名是“Million Dollar Baby”吗？或者是否应该使用原版名称？比如《百万英镑》在中文里通常翻译为《百万英镑》，但英文名可能不同，不过用户可能希望用原名，所以需要确认。

另外，检查是否有重复的电影，确保每个电影都是唯一的。例如，《阿甘正传》和《百万英镑》是不同的电影，所以应该分开列出。

最后，确保XML的结构正确，比如正确的标签闭合，属性正确使用等。比如，每个movie元素必须有<movie>和</movie>，并且title属性正确。
</think>

```xml
<movie title="阿甘正传" />
<movie title="百万英镑" />
<movie title="无问西楼" />
<movie title="当幸福来敲门" />
<movie title="老无所依" />
<movie title="好家伙！" />
<movie title="情书" />
```


In [5]:
parser = XMLOutputParser()
parser.invoke(output)

OutputParserException: Failed to parse XML format from completion <movie title="阿甘正传" />
<movie title="百万英镑" />
<movie title="无问西楼" />
<movie title="当幸福来敲门" />
<movie title="老无所依" />
<movie title="好家伙！" />
<movie title="情书" />. Got: junk after document element: line 2, column 0
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

In [6]:
parser = XMLOutputParser()
# format_instructions =  parser.get_format_instructions()
format_instructions = """响应以xml的结构返回，使用如下xml结构
```
<xml>
<movie>电影1</movie>
<movie>电影2</movie>
<xml>
```
"""
prompt = PromptTemplate(
    template="""{query}\n{format_instructions}""",
    input_variables=["query"],
    partial_variables={"format_instructions":format_instructions},
)

chain = prompt | model | parser

output = chain.invoke({"query": actor_query})
print(output)

OutputParserException: Failed to parse XML format from completion <xml>
<movie>阿甘正传</movie>
<movie>无问西楼</movie>
<movie>百万英镑</movie>
<movie>燃眉之战</movie>
<xml>. Got: no element found: line 6, column 5
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

In [7]:
parser = XMLOutputParser()
format_instructions =  parser.get_format_instructions()
format_instructions

'The output should be formatted as a XML file.\n1. Output should conform to the tags below.\n2. If tags are not given, make them on your own.\n3. Remember to always open and close all the tags.\n\nAs an example, for the tags ["foo", "bar", "baz"]:\n1. String "<foo>\n   <bar>\n      <baz></baz>\n   </bar>\n</foo>" is a well-formatted instance of the schema.\n2. String "<foo>\n   <bar>\n   </foo>" is a badly-formatted instance.\n3. String "<foo>\n   <tag>\n   </tag>\n</foo>" is a badly-formatted instance.\n\nHere are the output tags:\n```\nNone\n```'